### WandB Run Analysis

In [ ]:
from typing import Any

import pandas as pd
import wandb
from tqdm import tqdm
from vero_benchmarking.constants import DEFAULT_RESULTS_DIR

pd.set_option("display.float_format", "{:.2f}".format)

# Display key columns in our final processed dataframe
display_cols = [
    "run_id",
    "optimizer_scaffold",
    "policy_type",
    "task",
    "model",
    "session_id",
    "base_commit",
    "final_commit",
    "initial_score",
    "best_score",
    "num_evals",
    "initial_commit",
    "best_commit",
    "initial_error_rate",
    "best_error_rate",
]

Initialize the WandB API and list the runs

In [ ]:
# Initialize wandb API
api = wandb.Api()
project = "vero-icml-final"

# Fetch all runs from the project
runs = list(api.runs(project))
print(f"Found {len(runs)} runs")

Extract the relevant data from nested dictionaries in each Run object

In [ ]:
# Build consolidated dataframe: one row per run with summary + aggregated histories


def default_columns() -> list[str]:
    return ["score", "num_samples", "candidate_commit", "error_rate"]


def get_default_columns_map(prefix: str) -> dict[str, str]:
    cols = default_columns()
    return {f"{prefix}/{c}": c for c in cols}


def extract_history_metrics(hist_df: pd.DataFrame, column_map: dict[str, str] | str) -> list[dict]:
    """Extract metrics for a given prefix (train/validation/test) as list of dicts."""

    if isinstance(column_map, str):
        prefix = column_map
        column_map = get_default_columns_map(prefix)

    available = [c for c in column_map if c in hist_df.columns]
    if not available:
        return []
    subset = hist_df[available].dropna(how="all")
    records = []
    for _, row in subset.iterrows():
        record = {}
        for col, key in column_map.items():
            record[key] = row.get(col)

        # Only include if at least one value is not null
        if any(pd.notna(v) for v in record.values()):
            records.append(record)
    return records


# Extract nested config from vero-benchmarking-config


def get_nested(config: dict, *keys, default: Any = None) -> Any:
    val = config
    for k in keys:
        try:
            val = val.get(k)
        except KeyError:
            return default
        except AttributeError:
            print(f"AttributeError for {config} with keys {keys}")
    return val


# Build the full dataframe


def build_run_df_with_history(runs: list[wandb.Run]) -> pd.DataFrame:
    """Build a dataframe with history metrics for each run."""

    data = []
    for run in tqdm(runs):
        hist_df = run.history()
        row = {
            "run_id": run.id,
            "name": run.name,
            "state": run.state,
            "created_at": run.created_at,
            "config": dict(run.config),
            "summary": dict(run.summary),
            "best_results": dict(run.summary.get("best_results", {})),
            # Aggregated history metrics as lists
            "train_history": extract_history_metrics(hist_df, "train"),
            "validation_history": extract_history_metrics(hist_df, "validation"),
            "test_history": extract_history_metrics(hist_df, "test"),
        }
        data.append(row)

    df = pd.DataFrame(data)
    return df


# Use the nested getter to bring nested keys to the top level


def extract_primary_fields(df: pd.DataFrame) -> pd.DataFrame:
    extractions = {
        "base_branch": (
            "summary",
            "config",
            "base_branch",
        ),
        "base_commit": (
            "summary",
            "config",
            "base_commit",
        ),
        "final_commit": (
            "summary",
            "config",
            "final_commit",
        ),
        "model": (
            "summary",
            "config",
            "model",
        ),
        "session_id": (
            "summary",
            "config",
            "session_id",
        ),
        "optimizer_scaffold": ("config", "vero-benchmarking-config", "name"),
        "policy_type": ("config", "vero-benchmarking-config", "policy_type"),
        "task": ("config", "vero-benchmarking-config", "task", "task"),
    }

    for col, keys in extractions.items():
        src = keys[0]
        df[col] = df[src].apply(lambda x, k=keys[1:]: get_nested(x, *k))

Now actually perform the extraction and print out how many runs we have in total

In [ ]:
df = build_run_df_with_history(runs)
print(f"Loaded {len(df)} runs")
df.head()

In [ ]:
# Brings nested keys to the top level
extract_primary_fields(df)

We use test set history on most tasks, but for GAIA we use validation set history

In [ ]:
# Map tasks to the target history column
task_to_split_map = {
    "math": "test_history",
    "gpqa": "test_history",
    "simple_qa": "test_history",
    "gaia": "validation_history",
    "retail": "test_history",
}

df = df[df["task"].isin(task_to_split_map)].copy()
df["performance_dimension"] = df["task"].map(task_to_split_map)

Get the performance by extracting the initial and final commits along the performance dimension

In [ ]:
def extract_performance(row: pd.Series) -> dict:
    """Extract performance metrics from a row of the dataframe."""
    col = row["performance_dimension"]
    num_evals = len(row[col])

    if num_evals < 2:
        return {
            "initial_score": None,
            "best_score": None,
            "num_evals": num_evals,
            "initial_commit": None,
            "best_commit": None,
            "initial_error_rate": None,
            "best_error_rate": None,
        }

    base_commit = row["base_commit"]

    initial_score = None
    initial_commit = None
    best_score = None
    best_commit = None
    best_error_rate = None
    initial_error_rate = None

    for d in row[col]:

        commit = d.get("candidate_commit")
        if commit == base_commit:
            if initial_score is None or d["score"] > initial_score:
                initial_score = d["score"]
                initial_commit = commit
                initial_error_rate = d["error_rate"]
        elif commit is not None:
            if best_score is None or d["score"] > best_score:
                best_score = d["score"]
                best_commit = commit
                best_error_rate = d["error_rate"]
    return {
        "initial_score": initial_score,
        "best_score": best_score,
        "num_evals": num_evals,
        "initial_commit": initial_commit,
        "best_commit": best_commit,
        "initial_error_rate": initial_error_rate,
        "best_error_rate": best_error_rate,
    }


def check_row_quality(row: pd.Series, error_rate_threshold: float = 0.15) -> pd.DataFrame:
    tags = []
    if pd.isna(row["base_commit"]):
        tags.append("initial_commit_missing")

    if pd.isna(row["final_commit"]):
        tags.append("final_commit_missing")

    performance_dimension = row["performance_dimension"]
    history = row[performance_dimension]
    if len(history) < 2:
        tags.append("insufficient_history")

    if row["best_error_rate"] and row["best_error_rate"] > error_rate_threshold:
        tags.append("high_best_error_rate")

    if row["initial_error_rate"] and row["initial_error_rate"] > error_rate_threshold:
        tags.append("high_initial_error_rate")

    return tags


df[
    [
        "initial_score",
        "best_score",
        "num_evals",
        "initial_commit",
        "best_commit",
        "initial_error_rate",
        "best_error_rate",
    ]
] = df.apply(extract_performance, axis=1, result_type="expand")
df["quality_tags"] = df.apply(check_row_quality, axis=1)
df["bad_run"] = df["quality_tags"].apply(bool)

print("# of bad runs: ", df["bad_run"].sum())

In [ ]:
df[df["bad_run"]].head(2)

Remove any bad runs with missing data

In [ ]:
filtered_df = df[~df["bad_run"]].copy()

Ensure that we have a value for model - the default across runs was claude sonnet

In [ ]:
filtered_df[filtered_df["model"].isna()][["optimizer_scaffold", "task"]].value_counts()

In [ ]:
filtered_df["model"] = filtered_df["model"].fillna("anthropic/claude-sonnet-4-5-20250929")

In [ ]:
print(filtered_df.shape)

filtered_df[display_cols].head()

In [ ]:
filtered_df[["optimizer_scaffold", "task", "model"]].value_counts()

Ensure we have the same amount of data for each row

In [ ]:
# Keep only the 3 most recent runs per (optimizer_scaffold, task, model)
filtered_df = (
    filtered_df.sort_values("created_at", ascending=False)
    .groupby(["optimizer_scaffold", "task", "model"])
    .head(3)
)
print(f"Kept {len(filtered_df)} runs (top 3 most recent per group)")
filtered_df[["optimizer_scaffold", "task", "model"]].value_counts()

Since the initial agent is the same in all cases, we can get a more accurate number by taking an average over all scores for each config x task

In [ ]:
filtered_df["avg_initial_score_by_task"] = filtered_df.groupby("task")["initial_score"].transform(
    "mean"
)
filtered_df["max_initial_score_by_task"] = filtered_df.groupby("task")["initial_score"].transform(
    "max"
)

In [ ]:
filtered_df["lift"] = filtered_df["best_score"] - filtered_df["initial_score"]
filtered_df.sort_values(by="lift").head(3)

In [ ]:
filtered_df[display_cols].to_csv(DEFAULT_RESULTS_DIR / "benchmark_results.csv")
filtered_df.to_csv(DEFAULT_RESULTS_DIR / "benchmark_results_with_history.csv")

In [ ]:
df.task.value_counts()

Claude Code Pure runs ran without computing initial/final performance on validation set, so we need to repeat

In [ ]:
claude_code_pure_tasks = []

sub_df = df[(df.optimizer_scaffold == "claude-code-pure") & (df.task == "gaia")]

for idx, row in sub_df.iterrows():
    alias = "gpt-4.1-mini-2025-04-14"
    model = alias
    claude_code_pure_tasks.append(
        dict(
            task="gaia",
            commit=row["final_commit"],
            model=model,
            split="validation",
            is_initial_commit=True,
            model_alias=alias,
            optimizer_scaffold=row["optimizer_scaffold"],
            optimizer_model=row["model"],
        )
    )


claude_code_pure_tasks_df = pd.DataFrame(claude_code_pure_tasks)
claude_code_pure_tasks_df

In [ ]:
path_to_claude_code_gaia = DEFAULT_RESULTS_DIR / "claude_code_gaia"
path_to_claude_code_gaia.mkdir(parents=True, exist_ok=True)
claude_code_pure_tasks_df.to_csv(DEFAULT_RESULTS_DIR / "claude_code_gaia" / "manifest.csv")

In [ ]:
claude_code_pure_tasks_results_df = pd.read_parquet(path_to_claude_code_gaia / "summary.parquet")
claude_code_pure_gaia_mean_score = claude_code_pure_tasks_results_df.mean_score.mean()
claude_code_pure_gaia_max_score = claude_code_pure_tasks_results_df.mean_score.max()
print(claude_code_pure_gaia_mean_score)
print(claude_code_pure_gaia_max_score)

In [ ]:
filtered_df = pd.read_csv(DEFAULT_RESULTS_DIR / "benchmark_results_with_history.csv")

In [ ]:
iteration_aggregations = dict(
    num_runs=("run_id", "count"),
    avg_best_score=("best_score", "mean"),
    avg_initial_score=("avg_initial_score_by_task", "mean"),
    max_initial_score=("max_initial_score_by_task", "mean"),
    max_best_score=("best_score", "max"),
)


agg_results = (
    filtered_df.groupby(["optimizer_scaffold", "model", "task"])
    .agg(**iteration_aggregations)
    .sort_values(by=["optimizer_scaffold", "model", "task"], ascending=False)
)
agg_results["avg_lift"] = agg_results["avg_best_score"] - agg_results["avg_initial_score"]
agg_results["max_lift_over_average"] = (
    agg_results["max_best_score"] - agg_results["avg_initial_score"]
)
agg_results["max_lift_over_max"] = agg_results["max_best_score"] - agg_results["max_initial_score"]

task_agg_results = agg_results.groupby(["optimizer_scaffold", "model"]).agg(
    average_average_lift=("avg_lift", "mean"),
    average_max_lift_over_average=("max_lift_over_average", "mean"),
    average_max_lift_over_max=("max_lift_over_max", "mean"),
)

In [ ]:
agg_results

In [ ]:
agg_results[
    [
        "num_runs",
        "avg_initial_score",
        "avg_best_score",
        "max_best_score",
        "avg_lift",
        "max_lift_over_average",
    ]
]

In [ ]:
task_agg_results

In [ ]:
agg_results.to_csv(DEFAULT_RESULTS_DIR / "benchmark_iteration_aggregates.csv")
task_agg_results.to_csv(DEFAULT_RESULTS_DIR / "benchmark_task_aggregates.csv")

In [ ]:
# Filter by task, model, and optimizer scaffold for robustness experiments

filtered_df = pd.read_csv(DEFAULT_RESULTS_DIR / "benchmark_results_with_history.csv")

model_filter = filtered_df["model"].str.contains("sonnet|gpt", case=False)
task_filter = filtered_df["task"].isin(["retail", "gpqa", "simple_qa", "gaia"])
scaffold_filter = filtered_df["optimizer_scaffold"].isin(["vero-orchestrator-cookbook"])

robustness_df = filtered_df[model_filter & task_filter & scaffold_filter]

# Get best row per (task, model)
robustness_df = robustness_df.loc[robustness_df.groupby(["task", "model"])["best_score"].idxmax()]

robustness_df = robustness_df[display_cols]
robustness_df

In [ ]:
# Build the manifest of experiments to run for robustness experiments

rows = []

model_aliases = {
    "gpt-4.1",
    "anthropic/claude-sonnet-4-5-20250929",
    "llmengine/qwen3-30b-a3b-instruct-2507",
    "llmengine/qwen3-4b-instruct-2507",
    "gpt-5-mini-2025-08-07",
    "anthropic/claude-haiku-4-5-20251001",
    "gemini/gemini-2.5-flash",
}
vero_task_to_optimization_task = {
    "gpqa": "gpqa-nosplit",
    "simple_qa": "simpleqa",
    "retail": "tau-bench",
    "gaia": "gaia",
}


def get_split_for_task(task: str) -> str:
    if task == "gaia":
        return "validation"
    return "test"


def map_alias_to_model_endpoint(alias: str, task: str) -> str:

    if task in ["gpqa", "gaia"]:
        if alias.startswith("anthropic") or alias.startswith("gemini"):
            return f"litellm/{alias}"
        if alias.startswith("llmengine"):
            return f"litellm/openai/{alias}"

    if task in ["simple_qa", "retail"]:
        if alias.startswith("llmengine"):
            return f"openai/{alias}"

    return alias


for idx, row in robustness_df.iterrows():
    split = get_split_for_task(row["task"])
    for alias in model_aliases:
        model = map_alias_to_model_endpoint(alias, row["task"])
        rows.append(
            dict(
                task=vero_task_to_optimization_task[row["task"]],
                commit=row["initial_commit"],
                model=model,
                split=split,
                is_initial_commit=True,
                model_alias=alias,
                optimizer_scaffold=row["optimizer_scaffold"],
                optimizer_model=row["model"],
            )
        )
        rows.append(
            dict(
                task=vero_task_to_optimization_task[row["task"]],
                commit=row["best_commit"],
                split=split,
                model=model,
                is_initial_commit=False,
                model_alias=alias,
                optimizer_scaffold=row["optimizer_scaffold"],
                optimizer_model=row["model"],
            )
        )

robustness_manifest = pd.DataFrame(rows)

In [ ]:
robustness_df

In [ ]:
robustness_df.to_csv(DEFAULT_RESULTS_DIR / "benchmark_experiment_source_configs.csv")

robustness_experiments_dir = DEFAULT_RESULTS_DIR / "robustness_experiment_final"
robustness_experiments_dir.mkdir(parents=True, exist_ok=True)
robustness_manifest.to_csv(robustness_experiments_dir / "manifest.csv")

In [ ]:
robustness_manifest

In [ ]:
# Create simplified pivoted table for paper:
# - 1 row for baseline (initial scores)
# - 1 row per config showing avg (max) best scores
# - 1 column per task + average column


def format_with_max(avg_val, max_val):
    """Format as 'avg (max)'"""
    return f"{avg_val:.2f} ({max_val:.2f})"


# Reset index to work with columns
agg_reset = agg_results.reset_index()

# Create formatted "best" column: avg (max)
agg_reset["best_formatted"] = agg_reset.apply(
    lambda row: format_with_max(row["avg_best_score"], row["max_best_score"]), axis=1
)

# Pivot to get tasks as columns - just the best scores
pivot_best = agg_reset.pivot_table(
    index=["optimizer_scaffold", "model"], columns="task", values="best_formatted", aggfunc="first"
)

# Also pivot numeric values for computing averages
pivot_avg_numeric = agg_reset.pivot_table(
    index=["optimizer_scaffold", "model"], columns="task", values="avg_best_score", aggfunc="first"
)
pivot_max_numeric = agg_reset.pivot_table(
    index=["optimizer_scaffold", "model"], columns="task", values="max_best_score", aggfunc="first"
)

# Fill in Claude Code Pure GAIA data (from separate re-run)
claude_pure_idx = ("claude-code-pure", "anthropic/claude-sonnet-4-5-20250929")
if claude_pure_idx in pivot_avg_numeric.index:
    pivot_avg_numeric.loc[claude_pure_idx, "gaia"] = claude_code_pure_gaia_mean_score
    pivot_max_numeric.loc[claude_pure_idx, "gaia"] = claude_code_pure_gaia_max_score
    pivot_best.loc[claude_pure_idx, "gaia"] = format_with_max(
        claude_code_pure_gaia_mean_score, claude_code_pure_gaia_max_score
    )

# Compute row-wise average (across tasks) - now includes filled GAIA data
row_avg = pivot_avg_numeric.mean(axis=1)
row_max_avg = pivot_max_numeric.mean(axis=1)

# Get baseline initial scores (should be same across configs for each task)
baseline_scores = agg_reset.groupby("task")["avg_initial_score"].first()

# Create the paper table
paper_table = pivot_best.copy()

# Add average column
paper_table["Average"] = [format_with_max(a, m) for a, m in zip(row_avg, row_max_avg)]

# Add baseline row at the top
baseline_avg = baseline_scores.mean()
baseline_row = pd.DataFrame(
    {
        **{task: [f"{baseline_scores[task]:.2f}"] for task in pivot_best.columns},
        "Average": [f"{baseline_avg:.2f}"],
    },
    index=pd.MultiIndex.from_tuples([("Baseline", "-")], names=["optimizer_scaffold", "model"]),
)
paper_table = pd.concat([baseline_row, paper_table])

# Display
print("Paper Results Table:")
print("(Baseline row shows initial score, other rows show avg best score (max best score))")
paper_table

In [ ]:
paper_table

In [ ]:
# Save paper table to CSV
paper_table.to_csv(DEFAULT_RESULTS_DIR / "paper_table_1.csv")
print(f"Saved to {DEFAULT_RESULTS_DIR / 'paper_table_1.csv'}")